# Label note sections (rules-first, pre-publish)

Assigns **section boundaries on the unchanged raw note** using `section_lexicon.json` (no LLM). Always upserts section JSON; warnings only.

**Workflow:** Config → Load cases → **Single-case pilot** → Batch → Summary

Prerequisites: migration applied, `setup_cohort.ipynb` run, optional `inspect_note_headings.ipynb` for lexicon review.

```bash
pip install -r notebooks/requirements.txt
```

In [11]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from supabase import create_client

NOTEBOOKS_DIR = Path.cwd() if (Path.cwd() / "lib" / "note_polish.py").exists() else Path.cwd() / "notebooks"
LIB_DIR = str(NOTEBOOKS_DIR / "lib")
if LIB_DIR not in sys.path:
    sys.path.insert(0, LIB_DIR)

# Drop cached modules so edits to note_polish.py are picked up without restarting the kernel.
for _mod in ("note_polish", "section_detect", "section_types"):
    sys.modules.pop(_mod, None)

from note_polish import (
    PROMPT_VERSION,
    build_case_update,
    build_run_record,
    label_case_pair,
    result_preview,
    should_skip_case,
)
from section_detect import load_lexicon

load_dotenv(NOTEBOOKS_DIR / ".env")
load_dotenv(Path(".env"))

import os

SUPABASE_URL = os.environ["SUPABASE_URL"]
SUPABASE_SERVICE_ROLE_KEY = os.environ["SUPABASE_SERVICE_ROLE_KEY"]
DRY_RUN = False
TEST_ROW_ID = os.getenv("TEST_ROW_ID", "75796")

sb = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)
LEXICON = load_lexicon()

RUN_DIR = NOTEBOOKS_DIR / "polish_runs" / pd.Timestamp.now("UTC").strftime("%Y%m%dT%H%M%SZ")
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("Prompt:", PROMPT_VERSION)
print("Lexicon entries:", len(LEXICON.get("canonical", [])))
print("dry_run:", DRY_RUN)
print("Run logs:", RUN_DIR)

Prompt: sections-rules-v1
Lexicon entries: 16
dry_run: False
Run logs: c:\Users\Lough\Desktop\Research\[Yale] Postdoctoral Research\Readmission\notebooks\polish_runs\20260530T213838Z


## Load assigned cases

In [6]:
assignments = sb.table("assignments").select("row_id").execute().data or []
row_ids = sorted({str(r["row_id"]) for r in assignments})
print(f"Assigned cases: {len(row_ids)}")

if not row_ids:
    raise RuntimeError(
        "No assigned cases — run setup_cohort.ipynb first and verify notebooks/.env SUPABASE_URL"
    )

cases = []
for rid in row_ids:
    row = sb.table("cases").select("*").eq("row_id", rid).single().execute().data
    cases.append(row)

cases_df = pd.DataFrame(cases)
cases_df[["row_id", "patient_identifier", "note_canonical_version", "note_version_hash"]].head(10)

Assigned cases: 60


,row_id,patient_identifier,note_canonical_version,note_version_hash
0,137,subject_7107_index_148024_readmit_104098,raw_v0,fnv1a-8de52147
1,139,subject_7192_index_128007_readmit_188914,raw_v0,fnv1a-d18113f0
2,141,subject_7251_index_196370_readmit_189182,raw_v0,fnv1a-08286e59
3,213,subject_10675_index_195182_readmit_141668,raw_v0,fnv1a-684a4915
4,215,subject_10689_index_116873_readmit_109618,raw_v0,fnv1a-e883746f
5,218,subject_10774_index_173586_readmit_197363,raw_v0,fnv1a-495f9871
6,23,subject_1354_index_135614_readmit_144830,raw_v0,fnv1a-8fbb0cb6
7,235,subject_11318_index_111001_readmit_151558,raw_v0,fnv1a-d9e7acb7
8,244,subject_11512_index_136412_readmit_170315,raw_v0,fnv1a-d9515c95
9,265,subject_13329_index_160303_readmit_194160,raw_v0,fnv1a-ebffd06a


## Single-case pilot

Set `TEST_ROW_ID` in the config cell (default `137`). Saves `pilot_{row_id}.json` with sections + warnings.

In [14]:
pilot_row = next(r for r in cases if str(r["row_id"]) == TEST_ROW_ID)
index_raw = pilot_row.get("index_discharge_summary") or ""
readmit_raw = pilot_row.get("readmit_discharge_summary") or ""

print(f"Labeling case {TEST_ROW_ID} with rules engine ...")
pilot_index, pilot_readmit, pilot_meta = label_case_pair(index_raw, readmit_raw)
pilot_update = build_case_update(pilot_index, pilot_readmit, pilot_meta)
pilot_record = build_run_record(
    TEST_ROW_ID,
    index_raw,
    readmit_raw,
    pilot_index,
    pilot_readmit,
    pilot_meta,
    pilot_update,
)

pilot_path = RUN_DIR / f"pilot_{TEST_ROW_ID}.json"
pilot_path.write_text(json.dumps(pilot_record, indent=2), encoding="utf-8")

print("Warnings:", pilot_meta["warning_count"])
print("Index:", pilot_index.section_source, len(pilot_index.sections), "sections", pilot_index.warnings)
print("Readmit:", pilot_readmit.section_source, len(pilot_readmit.sections), "sections", pilot_readmit.warnings)
print("Saved →", pilot_path)

for label, res, raw in [
    ("INDEX", pilot_index, index_raw),
    ("READMIT", pilot_readmit, readmit_raw),
]:
    print(f"\n=== {label} sections ===")
    for s in res.sections:
        preview = raw[s.startChar : min(s.endChar, s.startChar + 80)].replace("\n", " ")
        print(f"  {s.id:8} {s.title:28} [{s.startChar}:{s.endChar}]  {preview[:70]}...")

StopIteration: 

In [4]:
# Reload saved pilot JSON (no API)
INSPECT_ROW_ID = TEST_ROW_ID
log_path = RUN_DIR / f"pilot_{INSPECT_ROW_ID}.json"
if log_path.exists():
    saved = json.loads(log_path.read_text(encoding="utf-8"))
    print("Loaded:", log_path)
    print("Warnings:", saved["meta"].get("warning_count"))
    for note in ("index", "readmission"):
        p = saved["preview"][note]
        print(f"\n--- {note} ({p['section_source']}) — {len(p['sections'])} sections ---")
        for sp in p.get("section_previews", [])[:8]:
            print(sp["id"], sp["title"], sp["preview"][:60])

Loaded: c:\Users\Lough\Desktop\Research\[Yale] Postdoctoral Research\Readmission\notebooks\polish_runs\20260530T213258Z\pilot_137.json
Warnings: 3

--- index (rules) — 12 sections ---
sec-001 Preamble Name:  [**Known lastname 5378**], [**Known firstname **]    
sec-002 Addendum 
ADDENDUM:   Please add to previous dictation of discharge
s
sec-003 Family History 
FAMILY HISTORY:   The patient has a father and brother with
sec-004 Social History 
SOCIAL HISTORY:   History of tobacco use, rare alcohol use.
sec-005 Physical Exam 
PHYSICAL EXAMINATION:   On admission, temperature of 102.0

sec-006 Laboratory 
LABORATORY:  On admission are notable for a white blood cel
sec-007 Brief Hospital Course 
HOSPITAL COURSE:
1.  HYPERTENSION:  The patient was hypoten
sec-008 Discharge Condition 
CONDITION ON DISCHARGE:   Hemodynamically stable, ambulatin

--- readmission (rules) — 4 sections ---
sec-001 Preamble Admission Date:  [**2145-4-27**]       Discharge Date:  [**2
sec-002 Discharge Diagnosis 


## Batch label loop

Always writes sections. Set `DRY_RUN=false` in `.env` to upsert Supabase.

In [15]:
results = []

for row in cases:
    row_id = str(row["row_id"])
    if should_skip_case(row):
        print(f"Skip {row_id} (already labeled {PROMPT_VERSION})")
        continue

    index_raw = row.get("index_discharge_summary") or ""
    readmit_raw = row.get("readmit_discharge_summary") or ""

    index_res, readmit_res, formatting_meta = label_case_pair(index_raw, readmit_raw)

    update = build_case_update(index_res, readmit_res, formatting_meta)
    record = build_run_record(
        row_id,
        index_raw,
        readmit_raw,
        index_res,
        readmit_res,
        formatting_meta,
        update,
    )

    out_path = RUN_DIR / f"{row_id}.json"
    out_path.write_text(json.dumps(record, indent=2), encoding="utf-8")

    if not DRY_RUN:
        sb.table("cases").update(update).eq("row_id", row_id).execute()

    results.append({
        "row_id": row_id,
        "index_source": index_res.section_source,
        "readmit_source": readmit_res.section_source,
        "index_sections": len(index_res.sections),
        "readmit_sections": len(readmit_res.sections),
        "index_chars": len(index_raw),
        "readmit_chars": len(readmit_raw),
        "warnings": formatting_meta["warning_count"],
        "log": str(out_path),
    })
    print(
        row_id,
        f"idx={index_res.section_source}/{len(index_res.sections)}",
        f"rdm={readmit_res.section_source}/{len(readmit_res.sections)}",
        f"warn={formatting_meta['warning_count']}",
    )

summary_df = pd.DataFrame(results)
summary_df

137 idx=rules/12 rdm=rules/4 warn=3
139 idx=rules/19 rdm=rules/21 warn=4
141 idx=rules/21 rdm=rules/18 warn=3
213 idx=rules/25 rdm=rules/22 warn=21
215 idx=rules/19 rdm=rules/17 warn=2
218 idx=rules/39 rdm=rules/28 warn=32
23 idx=rules/28 rdm=rules/37 warn=30
235 idx=rules/16 rdm=rules/17 warn=1
244 idx=rules/18 rdm=rules/8 warn=1
265 idx=rules/37 rdm=rules/36 warn=35
266 idx=rules/20 rdm=rules/23 warn=7
298 idx=rules/18 rdm=rules/20 warn=4
30 idx=rules/2 rdm=rules/1 warn=1
302 idx=rules/11 rdm=rules/3 warn=4
305 idx=rules/19 rdm=rules/22 warn=6
312 idx=rules/25 rdm=rules/19 warn=10
338 idx=rules/6 rdm=rules/2 warn=3
342 idx=rules/25 rdm=rules/26 warn=15
346 idx=rules/20 rdm=rules/35 warn=19
350 idx=rules/36 rdm=rules/21 warn=23
359 idx=rules/4 rdm=rules/11 warn=4
370 idx=rules/22 rdm=rules/19 warn=6
39 idx=rules/18 rdm=rules/21 warn=4
396 idx=rules/22 rdm=rules/27 warn=17
429 idx=rules/19 rdm=rules/3 warn=2
439 idx=rules/23 rdm=rules/19 warn=7
457 idx=rules/39 rdm=rules/21 warn=27
481

,row_id,index_source,readmit_source,index_sections,readmit_sections,index_chars,readmit_chars,warnings,log
0,137,rules,rules,12,4,10962,3235,3,c:\Users\Lough\Desktop\Research\[Yale] Postdoc...
1,139,rules,rules,19,21,10199,9767,4,c:\Users\Lough\Desktop\Research\[Yale] Postdoc...
2,141,rules,rules,21,18,11767,8249,3,c:\Users\Lough\Desktop\Research\[Yale] Postdoc...
3,213,rules,rules,25,22,21056,17638,21,c:\Users\Lough\Desktop\Research\[Yale] Postdoc...
4,215,rules,rules,19,17,15997,8045,2,c:\Users\Lough\Desktop\Research\[Yale] Postdoc...
5,218,rules,rules,39,28,24817,7076,32,c:\Users\Lough\Desktop\Research\[Yale] Postdoc...
6,23,rules,rules,28,37,18366,21293,30,c:\Users\Lough\Desktop\Research\[Yale] Postdoc...
7,235,rules,rules,16,17,9883,8428,1,c:\Users\Lough\Desktop\Research\[Yale] Postdoc...
8,244,rules,rules,18,8,11462,3295,1,c:\Users\Lough\Desktop\Research\[Yale] Postdoc...
9,265,rules,rules,37,36,20814,23222,35,c:\Users\Lough\Desktop\Research\[Yale] Postdoc...


## Summary

In [16]:
if "summary_df" not in dir() or len(summary_df) == 0:
    print("No cases processed — run the batch loop first.")
else:
    print("Index sources:", summary_df["index_source"].value_counts().to_dict())
    print("Readmit sources:", summary_df["readmit_source"].value_counts().to_dict())
    print("Cases with warnings:", int((summary_df["warnings"] > 0).sum()))
    print("\nIndex section count histogram:")
    print(summary_df["index_sections"].value_counts().sort_index())
    print("\nReadmit section count histogram:")
    print(summary_df["readmit_sections"].value_counts().sort_index())
    long_single = summary_df[
        (summary_df["index_sections"] <= 1) & (summary_df["index_chars"] >= 1500)
    ]
    if len(long_single):
        print("\nLong index notes still single-section:")
        display(long_single[["row_id", "index_sections", "index_chars", "warnings"]])
    empty = summary_df[
        (summary_df["index_sections"] == 0) | (summary_df["readmit_sections"] == 0)
    ]
    if len(empty):
        display(empty)
    else:
        print("All processed cases have sections on both notes.")

Index sources: {'rules': 59, 'empty': 1}
Readmit sources: {'rules': 60}
Cases with warnings: 60

Index section count histogram:
index_sections
0     1
2     1
4     1
6     1
7     1
11    2
12    1
16    1
18    3
19    8
20    3
21    3
22    2
23    4
24    1
25    5
26    5
28    1
29    3
33    1
34    1
35    1
36    1
37    1
38    1
39    4
40    2
42    1
Name: count, dtype: int64

Readmit section count histogram:
readmit_sections
1     2
2     2
3     4
4     2
8     1
11    1
15    1
17    3
18    3
19    4
20    5
21    5
22    3
23    4
24    1
25    1
26    1
27    1
28    3
29    1
30    2
31    2
33    1
34    1
35    1
36    2
37    1
41    1
50    1
Name: count, dtype: int64


,row_id,index_source,readmit_source,index_sections,readmit_sections,index_chars,readmit_chars,warnings,log
30,490,empty,rules,0,2,0,1027,2,c:\Users\Lough\Desktop\Research\[Yale] Postdoc...


## Export full-cohort sections parquet

Writes `src/data/readmit_30d_sections.parquet` for every row in the source cohort (not only assigned cases). Requires `src/data/readmit_30d.parquet`.

In [17]:
from export_sections_parquet import export_sections_parquet

REPO_ROOT = NOTEBOOKS_DIR.parent if NOTEBOOKS_DIR.name == "notebooks" else NOTEBOOKS_DIR
SOURCE_PARQUET = Path(os.getenv("PARQUET_PATH", REPO_ROOT / "src" / "data" / "readmit_30d.parquet"))
OUTPUT_PARQUET = Path(os.getenv("SECTIONS_PARQUET_PATH", REPO_ROOT / "src" / "data" / "readmit_30d_sections.parquet"))
EXPORT_LIMIT = os.getenv("EXPORT_SECTIONS_LIMIT")
limit = int(EXPORT_LIMIT) if EXPORT_LIMIT else None

if not SOURCE_PARQUET.exists():
    raise FileNotFoundError(f"Source parquet not found: {SOURCE_PARQUET}")

sections_df = export_sections_parquet(SOURCE_PARQUET, OUTPUT_PARQUET, row_limit=limit)
print("Wrote", len(sections_df), "rows ->", OUTPUT_PARQUET)
sections_df[["row_id", "index_section_count", "readmit_section_count", "note_enrichment_version"]].head()

Wrote 908 rows -> c:\Users\Lough\Desktop\Research\[Yale] Postdoctoral Research\Readmission\src\data\readmit_30d_sections.parquet


,row_id,index_section_count,readmit_section_count,note_enrichment_version
0,0,19,3,sections-rules-v1
1,1,10,2,sections-rules-v1
2,2,19,24,sections-rules-v1
3,3,24,25,sections-rules-v1
4,4,36,20,sections-rules-v1
